# Test-Region Calculations

* Import
* Data preparation
* Calculations

In [ ]:
import arcpy
import rasterio
import os

from arcpy.sa import *
from arcpy.ia import *
from arcpy.da import *
from rasterio.plot import show_hist, show
from matplotlib import pyplot

%matplotlib inline 


In [ ]:
# Paths: # toDo -> check in which grid polygone the aoi is + completely new inp paths <-> saves time and computing power 

### Input
aoi_poly = r"F:\newData\Kassel\ka_shp\aoi_ka.shp"

buff_dist_build = "10 Meters"
buff_dist_quarry = "10 Meters"
buff_dist_water = "10 Meters"
buff_dist_roads = "10 Meters"
buff_dist_railw = "10 Meters"

# Difference-Rasters / LIDAR
diff_rast_1914 = r"F:\newData\testworkspace.gdb\copyraster1914"
diff_rast_2119 = r"F:\newData\testworkspace.gdb\copyraster2119_2"

# BBD-Data / PSI
psi_los_asce = r"F:\Umwelt 4.0\BBD_Update_2021_Hessen.gdb\BBD_2021_PSI_LOS_asce_mittlere_Geschwindigkeit"
psi_los_desc = r"F:\Umwelt 4.0\BBD_Update_2021_Hessen.gdb\BBD_2021_PSI_LOS_desc_mittlere_Geschwindigkeit"
psi_east_west = r"F:\Umwelt 4.0\BBD_Update_2021_Hessen.gdb\BBD_2021_PSI_Ost_West"
psi_vertical = r"F:\Umwelt 4.0\BBD_Update_2021_Hessen.gdb\BBD_2021_PSI_Vertikal"

# OSM-Data
osm_build = r"F:\newData\hesse_shp\gis_osm_buildings_a_free_1.shp"
osm_landu = r"F:\newData\hesse_shp\gis_osm_landuse_a_free_1.shp"
osm_water = r"F:\newData\hesse_shp\gis_osm_water_a_free_1.shp"
osm_railw = r"F:\newData\hesse_shp\gis_osm_railways_free_1.shp"
osm_roads = r"F:\newData\hesse_shp\gis_osm_roads_free_1.shp"

# Subsidence - Data
lyr_subsidence = r"F:\Umwelt 4.0\Setzungsempfindliche_Schichten.lpk"
gdb_subsidence = r"F:\Umwelt 4.0\setzungen.gdb"

### Output

# Folder Paths
folder_of = r"F:\newData\Kassel"
folder_of_rast = r"F:\newData\Kassel\ka_rast"
folder_of_shp = r"F:\newData\Kassel\ka_shp"
folder_of_psi = r"F:\newData\Kassel\ka_psi"
folder_of_subs = r"F:\newData\Kassel\ka_subs"


# Diff files
out_diff1914 = r"ka_diffrast_1914.tif"  
out_diff2119 = r"ka_diffrast_2119.tif"

mask_diff_1914 = r"ka_maskrast_1914.tif"
mask_diff_2119 = r"ka_maskrast_2119.tif" 

# BBD files

of_psi_los_asce = r"ka_psi_los_asce.shp"
of_psi_los_desc = r"ka_psi_los_desc.shp"
of_psi_east_west = r"ka_psi_east_west.shp"
of_psi_vertical = r"ka_psi_vertical.shp"

# OSM files

inp_fc_buildings = r"ka_osm_build.shp"
inp_fc_landuse = r"ka_osm_landu.shp"
inp_fc_water = r"ka_osm_water.shp"

inp_fc_railways = r"ka_osm_railw.shp"
inp_fc_roads = r"ka_osm_road.shp"

inp_fc_quarry = r"ka_osm_quarry.shp"
inp_fc_mainroads = "ka_osm_mroad.shp"

# buffer & erase frame

buffer_buildings = r"ka_buf{}_build.shp".format(buff_dist_build[0:2])
buffer_quarry = r"ka_buf{}_quarry.shp".format(buff_dist_quarry[0:2])
buffer_water = r"ka_buf{}_water.shp".format(buff_dist_water[0:2])
buffer_mainroads = r"ka_buf{}_mroads.shp".format(buff_dist_roads[0:2])
buffer_railways = r"ka_buf{}_railw.shp".format(buff_dist_railw[0:2])

buffer_complete = r"ka_cbuf.shp"
buffer_complete_dis = r"ka_dcbuf.shp"

mask_aoi = r"ka_erased_aoi.shp"



In [ ]:
# raster clipped - 1914
diff1914_clip = Clip(diff_rast_1914, aoi_poly) 
diff1914_clip.save(r"{}\{}".format(folder_of_rast, out_diff1914))

In [ ]:
# raster clipped - 2119
diff2119_clip = Clip(diff_rast_2119, aoi_poly) 
diff2119_clip.save(r"{}\{}".format(folder_of_rast, out_diff2119))

In [ ]:
# Clip all OSM-Data 

In [ ]:
arcpy.analysis.Clip(osm_build, aoi_poly, r"{}\{}".format(folder_of_shp,inp_fc_buildings))

In [ ]:
arcpy.analysis.Clip(osm_landu, aoi_poly, r"{}\{}".format(folder_of_shp,inp_fc_landuse))

In [ ]:
arcpy.analysis.Clip(osm_water, aoi_poly, r"{}\{}".format(folder_of_shp,inp_fc_water))

In [ ]:
arcpy.analysis.Clip(osm_railw, aoi_poly, r"{}\{}".format(folder_of_shp,inp_fc_railways))

In [ ]:
arcpy.analysis.Clip(osm_roads, aoi_poly, r"{}\{}".format(folder_of_shp,inp_fc_roads))

In [ ]:
# Extract Quarries | main roads

In [ ]:
arcpy.management.CopyFeatures(r"{}\{}".format(folder_of_shp,inp_fc_landuse), r"{}\{}".format(folder_of_shp,inp_fc_quarry), '', None, None, None)

field_name = "fclass"

with arcpy.da.UpdateCursor(r"{}\{}".format(folder_of_shp,inp_fc_quarry), [field_name]) as cursor:
    for row in cursor:
        if row[0] == 'quarry':
            continue
        else:
            cursor.deleteRow()

In [ ]:
arcpy.management.CopyFeatures(r"{}\{}".format(folder_of_shp,inp_fc_roads), r"{}\{}".format(folder_of_shp,inp_fc_mainroads), '', None, None, None)

field_name = "fclass"
road_list = ["motorway", "trunk", "primary", "secondary", "tertiary", "motorway_link", "trunk_link", "primary_link",  "secondary_link", "tertiary_link"]

with arcpy.da.UpdateCursor(r"{}\{}".format(folder_of_shp,inp_fc_mainroads), [field_name]) as cursor:
    for row in cursor:
        if row[0] in road_list:
            continue
        else:
            cursor.deleteRow()

In [ ]:
# Buffer, Mergen & Disslove

In [ ]:
arcpy.analysis.Buffer(r"{}\{}".format(folder_of_shp,inp_fc_buildings), r"{}\{}".format(folder_of_shp,buffer_buildings), buff_dist_build, "FULL", "ROUND", "ALL", None, "PLANAR")
arcpy.analysis.Buffer(r"{}\{}".format(folder_of_shp,inp_fc_quarry), r"{}\{}".format(folder_of_shp,buffer_quarry), buff_dist_quarry, "FULL", "ROUND", "ALL", None, "PLANAR")
arcpy.analysis.Buffer(r"{}\{}".format(folder_of_shp,inp_fc_water), r"{}\{}".format(folder_of_shp,buffer_water), buff_dist_water, "FULL", "ROUND", "ALL", None, "PLANAR")
arcpy.analysis.Buffer(r"{}\{}".format(folder_of_shp,inp_fc_mainroads), r"{}\{}".format(folder_of_shp,buffer_mainroads), buff_dist_roads, "FULL", "ROUND", "ALL", None, "PLANAR")
arcpy.analysis.Buffer(r"{}\{}".format(folder_of_shp,inp_fc_railways), r"{}\{}".format(folder_of_shp,buffer_railways), buff_dist_railw, "FULL", "ROUND", "ALL", None, "PLANAR")

In [ ]:
arcpy.management.Merge([r"{}\{}".format(folder_of_shp,buffer_buildings),
                        r"{}\{}".format(folder_of_shp,buffer_quarry),
                        r"{}\{}".format(folder_of_shp,buffer_water),
                        r"{}\{}".format(folder_of_shp,buffer_mainroads),
                        r"{}\{}".format(folder_of_shp,buffer_railways)], r"{}\{}".format(folder_of_shp,buffer_complete))

In [ ]:
arcpy.management.Dissolve(r"{}\{}".format(folder_of_shp,buffer_complete), r"{}\{}".format(folder_of_shp,buffer_complete_dis))

In [ ]:
# erase buffer shp from aoi

In [ ]:
arcpy.analysis.Erase(aoi_poly, r"{}\{}".format(folder_of_shp,buffer_complete_dis), r"{}\{}".format(folder_of_shp, mask_aoi), None) 

In [ ]:
# mask raster with mask_aoi

In [ ]:
masked_rast_1914 = arcpy.sa.ExtractByMask(r"{}\{}".format(folder_of_rast, out_diff1914), r"{}\{}".format(folder_of_shp, mask_aoi))
masked_rast_1914.save(r"{}\{}".format(folder_of_rast, mask_diff_1914))

In [ ]:
masked_rast_2119 = arcpy.sa.ExtractByMask(r"{}\{}".format(folder_of_rast, out_diff2119), r"{}\{}".format(folder_of_shp, mask_aoi))
masked_rast_2119.save(r"{}\{}".format(folder_of_rast, mask_diff_2119))

In [ ]:
 # calculate stats formasked difference rasters + reclassification

In [ ]:
inp_rec_1914 = Raster(r"{}\{}".format(folder_of_rast, mask_diff_1914))

# calculate stats
mdiff1914_min = inp_rec_1914.minimum
mdiff1914_max = inp_rec_1914.maximum
mdiff1914_mean = inp_rec_1914.mean
mdiff1914_std = inp_rec_1914.standardDeviation
print("Minimum: {0}, Maximum: {1}, Mean: {2}, Standard Deviation: {3}".format(mdiff1914_min, mdiff1914_max, mdiff1914_mean, mdiff1914_std))

In [ ]:
if mdiff1914_min > -0.35 or mdiff1914_max < 0.35:
    r1914_class10 = RemapRange([[-1000000000, -0.35,-4],[-0.35,-0.25,-3],[-0.25,-0.15,-2],[-0.15,-0.05,-1],[-0.05,0.05,"NODATA"],[0.05,0.15,1],[0.15,0.25,2],[0.25,0.35,3],[0.35, 1000000000,4]])
    r1914_class07 = RemapRange([[-1000000000, -0.35,"NODATA"],[-0.35,-0.25,-3],[-0.25,-0.15,-2],[-0.15,-0.05,-1],[-0.05,0.05,0],[0.05,0.15,1],[0.15,0.25,2],[0.25,0.35,3],[0.35, 1000000000,"NODATA"]])
else:
    r1914_class10 = RemapRange([[mdiff1914_min, -0.35,-4],[-0.35,-0.25,-3],[-0.25,-0.15,-2],[-0.15,-0.05,-1],[-0.05,0.05,"NODATA"],[0.05,0.15,1],[0.15,0.25,2],[0.25,0.35,3],[0.35, mdiff1914_max,4]])
    r1914_class07 = RemapRange([[mdiff1914_min, -0.35,"NODATA"],[-0.35,-0.25,-3],[-0.25,-0.15,-2],[-0.15,-0.05,-1],[-0.05,0.05,0],[0.05,0.15,1],[0.15,0.25,2],[0.25,0.35,3],[0.35, mdiff1914_max,"NODATA"]])
    
if mdiff1914_min > -0.4 or mdiff1914_max < 0.4:
    r1914_class03 = RemapRange([[-1000000000, -0.4,-5],[-0.4,-0.3,-4],[-0.3,-0.2,-3],[-0.2,-0.1,-2],[-0.1,0,-1],[0,0.1,1],[0.1,0.2,2],[0.2,0.3,3],[0.3,0.4,4],[0.4, 1000000000,5]])
    r1914_class01 = RemapRange([[-1000000000, -0.4,"NODATA"],[-0.4,-0.3,-4],[-0.3,-0.2,-3],[-0.2,-0.1,-2],[-0.1,0,-1],[0,0.1,1],[0.1,0.2,2],[0.2,0.3,3],[0.3,0.4,4],[0.4, 1000000000,"NODATA"]])
else:
    r1914_class03 = RemapRange([[mdiff1914_min, -0.4,-5],[-0.4,-0.3,-4],[-0.3,-0.2,-3],[-0.2,-0.1,-2],[-0.1,0,-1],[0,0.1,1],[0.1,0.2,2],[0.2,0.3,3],[0.3,0.4,4],[0.4, mdiff1914_max,5]])
    r1914_class01 = RemapRange([[mdiff1914_min, -0.4,"NODATA"],[-0.4,-0.3,-4],[-0.3,-0.2,-3],[-0.2,-0.1,-2],[-0.1,0,-1],[0,0.1,1],[0.1,0.2,2],[0.2,0.3,3],[0.3,0.4,4],[0.4, mdiff1914_max,"NODATA"]])

if mdiff1914_min > -0.7 or mdiff1914_max < 0.7:
    r1914_class09 = RemapRange([[-1000000000, -0.7,-4],[-0.7,-0.5,-3],[-0.5,-0.3,-2],[-0.3,-0.1,-1],[-0.1,0.1,"NODATA"],[0.1,0.3,1],[0.3,0.5,2],[0.5,0.7,3],[0.7, 1000000000,4]])
    r1914_class06 = RemapRange([[-1000000000, -0.7,"NODATA"],[-0.7,-0.5,-3],[-0.5,-0.3,-2],[-0.3,-0.1,-1],[-0.1,0.1,0],[0.1,0.3,1],[0.3,0.5,2],[0.5,0.7,3],[0.7, 1000000000,"NODATA"]])
else:
    r1914_class09 = RemapRange([[mdiff1914_min, -0.7,-4],[-0.7,-0.5,-3],[-0.5,-0.3,-2],[-0.3,-0.1,-1],[-0.1,0.1,"NODATA"],[0.1,0.3,1],[0.3,0.5,2],[0.5,0.7,3],[0.7, mdiff1914_max,4]])
    r1914_class06 = RemapRange([[mdiff1914_min, -0.7,"NODATA"],[-0.7,-0.5,-3],[-0.5,-0.3,-2],[-0.3,-0.1,-1],[-0.1,0.1,0],[0.1,0.3,1],[0.3,0.5,2],[0.5,0.7,3],[0.7, mdiff1914_max,"NODATA"]])
    
if mdiff1914_min > -1.0 or mdiff1914_max < 1.0:
    r1914_class04 = RemapRange([[-1000000000, -1,-5],[-1,-0.75,-4],[-0.75,-0.5,-3],[-0.5,-0.25,-2],[-0.25,0,-1],[0,0.25,1],[0.25,0.5,2],[0.5,0.75,3],[0.75,1,4],[1, 1000000000,5]])
    r1914_class02 = RemapRange([[-1000000000, -1,"NODATA"],[-1,-0.75,-4],[-0.75,-0.5,-3],[-0.5,-0.25,-2],[-0.25,0,-1],[0,0.25,1],[0.25,0.5,2],[0.5,0.75,3],[0.75,1,4],[1, 1000000000,"NODATA"]])
    
else:
    r1914_class04 = RemapRange([[mdiff1914_min, -1,-5],[-1,-0.75,-4],[-0.75,-0.5,-3],[-0.5,-0.25,-2],[-0.25,0,-1],[0,0.25,1],[0.25,0.5,2],[0.5,0.75,3],[0.75,1,4],[1, mdiff1914_max,5]])
    r1914_class02 = RemapRange([[mdiff1914_min, -1,"NODATA"],[-1,-0.75,-4],[-0.75,-0.5,-3],[-0.5,-0.25,-2],[-0.25,0,-1],[0,0.25,1],[0.25,0.5,2],[0.5,0.75,3],[0.75,1,4],[1, mdiff1914_max,"NODATA"]])

if mdiff1914_min > -1.75 or mdiff1914_max < 1.75:
    r1914_class08 = RemapRange([[-1000000000, -1.75,-4],[-1.75,-1.25,-3],[-1.25,-0.75,-2],[-0.75,-0.25,-1],[-0.25,0.25,"NODATA"],[0.25,0.75,1],[0.75,1.25,2],[1.25,1.75,3],[1.75, 1000000000,4]])
    r1914_class05 = RemapRange([[-1000000000, -1.75,"NODATA"],[-1.75,-1.25,-3],[-1.25,-0.75,-2],[-0.75,-0.25,-1],[-0.25,0.25,0],[0.25,0.75,1],[0.75,1.25,2],[1.25,1.75,3],[1.75, 1000000000,"NODATA"]])
else:
    r1914_class08 = RemapRange([[mdiff1914_min, -1.75,-4],[-1.75,-1.25,-3],[-1.25,-0.75,-2],[-0.75,-0.25,-1],[-0.25,0.25,"NODATA"],[0.25,0.75,1],[0.75,1.25,2],[1.25,1.75,3],[1.75, mdiff1914_max,4]])
    r1914_class05 = RemapRange([[mdiff1914_min, -1.75,"NODATA"],[-1.75,-1.25,-3],[-1.25,-0.75,-2],[-0.75,-0.25,-1],[-0.25,0.25,0],[0.25,0.75,1],[0.75,1.25,2],[1.25,1.75,3],[1.75, mdiff1914_max,"NODATA"]])

    
r1914_classes = [r1914_class02, r1914_class02, r1914_class03, r1914_class04, r1914_class05, r1914_class06, r1914_class07, r1914_class08, r1914_class09, r1914_class10]

In [ ]:
i = 0
for cl in r1914_classes:
    rc_rast1914 = Reclassify(r"{}\{}".format(folder_of_rast, mask_diff_1914), "Value", cl)
    rc_rast1914.save(r"{}\rc_rast1914_map{}.tif".format(folder_of_rast, i))
    i += 1

In [ ]:
inp_rec_2119 = Raster(r"{}\{}".format(folder_of_rast, mask_diff_2119))

# calculate stats
mdiff2119_min = inp_rec_2119.minimum
mdiff2119_max = inp_rec_2119.maximum
mdiff2119_mean = inp_rec_2119.mean
mdiff2119_std = inp_rec_2119.standardDeviation
print("Minimum: {0}, Maximum: {1}, Mean: {2}, Standard Deviation: {3}".format(mdiff2119_min, mdiff2119_max, mdiff2119_mean, mdiff2119_std))

In [ ]:
if mdiff2119_min > -0.35 or mdiff2119_max < 0.35:
    r2119_class10 = RemapRange([[-1000000000, -0.35,-4],[-0.35,-0.25,-3],[-0.25,-0.15,-2],[-0.15,-0.05,-1],[-0.05,0.05,"NODATA"],[0.05,0.15,1],[0.15,0.25,2],[0.25,0.35,3],[0.35, 1000000000,4]])
    r2119_class07 = RemapRange([[-1000000000, -0.35,"NODATA"],[-0.35,-0.25,-3],[-0.25,-0.15,-2],[-0.15,-0.05,-1],[-0.05,0.05,0],[0.05,0.15,1],[0.15,0.25,2],[0.25,0.35,3],[0.35, 1000000000,"NODATA"]])
else:
    r2119_class10 = RemapRange([[mdiff2119_min, -0.35,-4],[-0.35,-0.25,-3],[-0.25,-0.15,-2],[-0.15,-0.05,-1],[-0.05,0.05,"NODATA"],[0.05,0.15,1],[0.15,0.25,2],[0.25,0.35,3],[0.35, mdiff2119_max,4]])
    r2119_class07 = RemapRange([[mdiff2119_min, -0.35,"NODATA"],[-0.35,-0.25,-3],[-0.25,-0.15,-2],[-0.15,-0.05,-1],[-0.05,0.05,0],[0.05,0.15,1],[0.15,0.25,2],[0.25,0.35,3],[0.35, mdiff2119_max,"NODATA"]])
    
if mdiff2119_min > -0.4 or mdiff2119_max < 0.4:
    r2119_class03 = RemapRange([[-1000000000, -0.4,-5],[-0.4,-0.3,-4],[-0.3,-0.2,-3],[-0.2,-0.1,-2],[-0.1,0,-1],[0,0.1,1],[0.1,0.2,2],[0.2,0.3,3],[0.3,0.4,4],[0.4, 1000000000,5]])
    r2119_class01 = RemapRange([[-1000000000, -0.4,"NODATA"],[-0.4,-0.3,-4],[-0.3,-0.2,-3],[-0.2,-0.1,-2],[-0.1,0,-1],[0,0.1,1],[0.1,0.2,2],[0.2,0.3,3],[0.3,0.4,4],[0.4, 1000000000,"NODATA"]])
else:
    r2119_class03 = RemapRange([[mdiff2119_min, -0.4,-5],[-0.4,-0.3,-4],[-0.3,-0.2,-3],[-0.2,-0.1,-2],[-0.1,0,-1],[0,0.1,1],[0.1,0.2,2],[0.2,0.3,3],[0.3,0.4,4],[0.4, mdiff2119_max,5]])
    r2119_class01 = RemapRange([[mdiff2119_min, -0.4,"NODATA"],[-0.4,-0.3,-4],[-0.3,-0.2,-3],[-0.2,-0.1,-2],[-0.1,0,-1],[0,0.1,1],[0.1,0.2,2],[0.2,0.3,3],[0.3,0.4,4],[0.4, mdiff2119_max,"NODATA"]])

if mdiff2119_min > -0.7 or mdiff2119_max < 0.7:
    r2119_class09 = RemapRange([[-1000000000, -0.7,-4],[-0.7,-0.5,-3],[-0.5,-0.3,-2],[-0.3,-0.1,-1],[-0.1,0.1,"NODATA"],[0.1,0.3,1],[0.3,0.5,2],[0.5,0.7,3],[0.7, 1000000000,4]])
    r2119_class06 = RemapRange([[-1000000000, -0.7,"NODATA"],[-0.7,-0.5,-3],[-0.5,-0.3,-2],[-0.3,-0.1,-1],[-0.1,0.1,0],[0.1,0.3,1],[0.3,0.5,2],[0.5,0.7,3],[0.7, 1000000000,"NODATA"]])
else:
    r2119_class09 = RemapRange([[mdiff2119_min, -0.7,-4],[-0.7,-0.5,-3],[-0.5,-0.3,-2],[-0.3,-0.1,-1],[-0.1,0.1,"NODATA"],[0.1,0.3,1],[0.3,0.5,2],[0.5,0.7,3],[0.7, mdiff2119_max,4]])
    r2119_class06 = RemapRange([[mdiff2119_min, -0.7,"NODATA"],[-0.7,-0.5,-3],[-0.5,-0.3,-2],[-0.3,-0.1,-1],[-0.1,0.1,0],[0.1,0.3,1],[0.3,0.5,2],[0.5,0.7,3],[0.7, mdiff2119_max,"NODATA"]])
    
if mdiff2119_min > -1.0 or mdiff2119_max < 1.0:
    r2119_class04 = RemapRange([[-1000000000, -1,-5],[-1,-0.75,-4],[-0.75,-0.5,-3],[-0.5,-0.25,-2],[-0.25,0,-1],[0,0.25,1],[0.25,0.5,2],[0.5,0.75,3],[0.75,1,4],[1, 1000000000,5]])
    r2119_class02 = RemapRange([[-1000000000, -1,"NODATA"],[-1,-0.75,-4],[-0.75,-0.5,-3],[-0.5,-0.25,-2],[-0.25,0,-1],[0,0.25,1],[0.25,0.5,2],[0.5,0.75,3],[0.75,1,4],[1, 1000000000,"NODATA"]])
    
else:
    r2119_class04 = RemapRange([[mdiff2119_min, -1,-5],[-1,-0.75,-4],[-0.75,-0.5,-3],[-0.5,-0.25,-2],[-0.25,0,-1],[0,0.25,1],[0.25,0.5,2],[0.5,0.75,3],[0.75,1,4],[1, mdiff2119_max,5]])
    r2119_class02 = RemapRange([[mdiff2119_min, -1,"NODATA"],[-1,-0.75,-4],[-0.75,-0.5,-3],[-0.5,-0.25,-2],[-0.25,0,-1],[0,0.25,1],[0.25,0.5,2],[0.5,0.75,3],[0.75,1,4],[1, mdiff2119_max,"NODATA"]])

if mdiff2119_min > -1.75 or mdiff2119_max < 1.75:
    r2119_class08 = RemapRange([[-1000000000, -1.75,-4],[-1.75,-1.25,-3],[-1.25,-0.75,-2],[-0.75,-0.25,-1],[-0.25,0.25,"NODATA"],[0.25,0.75,1],[0.75,1.25,2],[1.25,1.75,3],[1.75, 1000000000,4]])
    r2119_class05 = RemapRange([[-1000000000, -1.75,"NODATA"],[-1.75,-1.25,-3],[-1.25,-0.75,-2],[-0.75,-0.25,-1],[-0.25,0.25,0],[0.25,0.75,1],[0.75,1.25,2],[1.25,1.75,3],[1.75, 1000000000,"NODATA"]])
else:
    r2119_class08 = RemapRange([[mdiff2119_min, -1.75,-4],[-1.75,-1.25,-3],[-1.25,-0.75,-2],[-0.75,-0.25,-1],[-0.25,0.25,"NODATA"],[0.25,0.75,1],[0.75,1.25,2],[1.25,1.75,3],[1.75, mdiff2119_max,4]])
    r2119_class05 = RemapRange([[mdiff2119_min, -1.75,"NODATA"],[-1.75,-1.25,-3],[-1.25,-0.75,-2],[-0.75,-0.25,-1],[-0.25,0.25,0],[0.25,0.75,1],[0.75,1.25,2],[1.25,1.75,3],[1.75, mdiff2119_max,"NODATA"]])

    
r2119_classes = [r2119_class01, r2119_class02, r2119_class03, r2119_class04, r2119_class05, r2119_class06, r2119_class07, r2119_class08, r2119_class09, r2119_class10]

In [ ]:
i = 0
for cl in r2119_classes:
    rc_rast2119 = Reclassify(r"{}\{}".format(folder_of_rast, mask_diff_2119), "Value", cl)
    rc_rast2119.save(r"{}\rc_rast2119_map{}.tif".format(folder_of_rast, i))
    i += 1

In [ ]:
# PSI data extraction

In [ ]:
arcpy.analysis.Clip(psi_los_asce, aoi_poly, r"{}\{}".format(folder_of_psi, of_psi_los_asce)) 

In [ ]:
arcpy.analysis.Clip(psi_los_desc, aoi_poly, r"{}\{}".format(folder_of_psi, of_psi_los_desc)) 

In [ ]:
arcpy.analysis.Clip(psi_east_west, aoi_poly, r"{}\{}".format(folder_of_psi, of_psi_east_west)) 

In [ ]:
arcpy.analysis.Clip(psi_vertical, aoi_poly, r"{}\{}".format(folder_of_psi, of_psi_vertical)) 

In [ ]:
# Subsidence Data extraction

In [ ]:
arcpy.management.ConsolidateLayer(lyr_subsidence, folder_of_subs)

In [ ]:
for dirpath, dirnames, filenames in arcpy.da.Walk(r"{}".format(folder_of_subs), datatype="FeatureClass"):
    for filename in filenames:
        inp_filepath = r"{0}\{1}".format(dirpath, filename)
        out_filepath = r"{0}\{1}_clip".format(dirpath, filename)
        
        arcpy.analysis.Clip(inp_filepath, aoi_poly, out_filepath)
        arcpy.management.Delete(inp_filepath)
        
        arcpy.management.Rename(out_filepath, inp_filepath)

        

In [ ]:
inp_rec_1914 = Raster(r"{}\{}".format(folder_of_rast, mask_diff_1914))

# calculate stats
mdiff1914_min = inp_rec_1914.minimum
mdiff1914_max = inp_rec_1914.maximum
mdiff1914_mean = inp_rec_1914.mean
mdiff1914_std = inp_rec_1914.standardDeviation
print("Minimum: {0}, Maximum: {1}, Mean: {2}, Standard Deviation: {3}".format(mdiff1914_min, mdiff1914_max, mdiff1914_mean, mdiff1914_std))

In [ ]:
r1914_classPSI = RemapRange([[-1000000000, -0.03,"NODATA"],[-0.03,-0.01,-6],[-0.01,-0.008,-5],[-0.008,-0.006,-4],[-0.006,-0.004,-3],[-0.004,-0.002,-2],[-0.002,0,-1],[0,0.002,1],[0.002,0.004,2],[0.004,0.006,3],[0.006,0.008,4],[0.008,0.01,5],[0.01,0.03,6],[0.03, 1000000000,"NODATA"]])

In [ ]:
rc_rast1914PSI = Reclassify(r"{}\{}".format(folder_of_rast, mask_diff_1914), "Value", r1914_classPSI)
rc_rast1914PSI.save(r"{}\rc_rast1914_mapPSI.tif".format(folder_of_rast))

In [ ]:
inp_rec_2119 = Raster(r"{}\{}".format(folder_of_rast, mask_diff_2119))

# calculate stats
mdiff2119_min = inp_rec_2119.minimum
mdiff2119_max = inp_rec_2119.maximum
mdiff2119_mean = inp_rec_2119.mean
mdiff2119_std = inp_rec_2119.standardDeviation
print("Minimum: {0}, Maximum: {1}, Mean: {2}, Standard Deviation: {3}".format(mdiff2119_min, mdiff2119_max, mdiff2119_mean, mdiff2119_std))

In [ ]:
r2119_classPSI = RemapRange([[-1000000000, -0.03,"NODATA"],[-0.03,-0.01,-6],[-0.01,-0.008,-5],[-0.008,-0.006,-4],[-0.006,-0.004,-3],[-0.004,-0.002,-2],[-0.002,0,-1],[0,0.002,1],[0.002,0.004,2],[0.004,0.006,3],[0.006,0.008,4],[0.008,0.01,5],[0.01,0.03,6],[0.03, 1000000000,"NODATA"]])

In [ ]:
rc_rast2119PSI = Reclassify(r"{}\{}".format(folder_of_rast, mask_diff_2119), "Value", r2119_classPSI)
rc_rast2119PSI.save(r"{}\rc_rast2119_mapPSI.tif".format(folder_of_rast))

In [ ]:
inp_rec_1914 = Raster(r"{}\{}".format(folder_of_rast, mask_diff_1914))

# calculate stats
mdiff1914_min = inp_rec_1914.minimum
mdiff1914_max = inp_rec_1914.maximum
mdiff1914_mean = inp_rec_1914.mean
mdiff1914_std = inp_rec_1914.standardDeviation
print("Minimum: {0}, Maximum: {1}, Mean: {2}, Standard Deviation: {3}".format(mdiff1914_min, mdiff1914_max, mdiff1914_mean, mdiff1914_std))

In [ ]:
r1914_class_IR05 = RemapRange([[-1000000000, -10,"NODATA"],[-10,-5,-6],[-5,-2.5,-5],[-2.5,-2,-4],[-2,-1.5,-3],[-1.5,-1,-2],[-1,-0.5,-1],[-0.5,0.5,"NODATA"],[0.5,1,1],[1,1.5,2],[1.5,2,3],[2,2.5,4],[2.5,5,5],[5,10,6],[10, 1000000000,"NODATA"]])

r1914_class_IR04 = RemapRange([[-1000000000, -10,"NODATA"],[-10,-8,-13],[-8,-6,-12],[-6,-5,-11],[-5,-4,-10],[-4,-3.6,-9],[-3.6,-3.2,-8],
                               [-3.2,-2.8,-7],[-2.8,-2.4,-6],[-2.4,-2.0,-5],[-2.0,-1.6,-4],[-1.6,-1.2,-3],[-1.2,-0.8,-2],[-0.8,-0.4,-1],[-0.4,0.4,"NODATA"],
                               [0.4,0.8,1],[0.8,1.2,2],[1.2,1.6,3],[1.6,2.0,4],[2.0,2.4,5],[2.4,2.8,6],[2.8,3.2,7],[3.2,3.6,8],
                               [3.6,4.0,9],[4.0,5,10],[5,6,11],[6,8,12],[8,10,13],[10, 1000000000,"NODATA"]])

r1914_class_IR03 = RemapRange([[-1000000000, -10,"NODATA"],[-10,-7.5,-9],[-7.5,-5,-8],[-5,-2.5,-7],[-2.5,-1.5,-6],[-1.5,-1.25,-5],[-1.25,-1,-4],[-1,-0.75,-3],[-0.75,-0.5,-2],[-0.5,-0.25,-1],[-0.25,0.25,"NODATA"],
                               [0.25,0.5,1],[0.5,0.75,2],[0.75,1,3],[1,1.25,4],[1.25,1.5,5],[1.5,2.5,6],[2.5,5,7],[5,7.5,8],[7.5,10,9],[10, 1000000000,"NODATA"]])

r1914_class_IR02 = RemapRange([[-1000000000, -10,"NODATA"],[-10,-8,-13],[-8,-6,-12],[-6,-4,-11],[-4,-2,-10],[-2,-1.8,-9],[-1.8,-1.6,-8],[-1.6,-1.4,-7],[-1.4,-1.2,-6],[-1.2,-1,-5],[-1,-0.8,-4],[-0.8,-0.6,-3],[-0.6,-0.4,-2],[-0.4,-0.2,-1],[-0.2,0.2,"NODATA"],
                               [0.2,0.4,1],[0.4,0.6,2],[0.6,0.8,3],[0.8,1,4],[1,1.2,5],[1.2,1.4,6],[1.4,1.6,7],[1.6,1.8,8],[1.8,2,9],[2,4,10],[4,6,11],[6,8,12],[8,10,13],[10, 1000000000,"NODATA"]])
                               
r1914_class_IR01 = RemapRange([[-1000000000, -10,"NODATA"],[-10,-8,-14],[-8,-6,-13],[-6,-4,-12],[-4,-3.5,-11],[-3.5,-3,-10],[-3,-2.5,-9],[-2.5,-2,-8],[-2,-1.5,-7],[-1.5,-1,-6],[-1,-0.5,-5],[-0.5,-0.4,-4],[-0.4,-0.3,-3],[-0.3,-0.2,-2],[-0.2,-0.1,-1],[-0.1,0.1,"NODATA"],
                               [0.1,0.2,1],[0.2,0.3,2],[0.3,0.4,3],[0.4,0.5,4],[0.5,1,5],[1,1.5,6],[1.5,2,7],[2,2.5,8],[2.5,3,9],[3,3.5,10],[3.5,4,11],[4,6,12],[6,8,13],[8,10,14],[10, 1000000000,"NODATA"]])

   

In [ ]:
r1914_classes = [r1914_class_IR01, r1914_class_IR02, r1914_class_IR03, r1914_class_IR04, r1914_class_IR05]

In [ ]:
i = 0
for cl in r1914_classes:
    rc_rast1914 = Reclassify(r"{}\{}".format(folder_of_rast, mask_diff_1914), "Value", cl)
    rc_rast1914.save(r"{}\rc_rast1914_mapIR{}.tif".format(folder_of_rast, i))
    i += 1

In [ ]:
inp_rec_2119 = Raster(r"{}\{}".format(folder_of_rast, mask_diff_2119))

# calculate stats
mdiff2119_min = inp_rec_2119.minimum
mdiff2119_max = inp_rec_2119.maximum
mdiff2119_mean = inp_rec_2119.mean
mdiff2119_std = inp_rec_2119.standardDeviation
print("Minimum: {0}, Maximum: {1}, Mean: {2}, Standard Deviation: {3}".format(mdiff2119_min, mdiff2119_max, mdiff2119_mean, mdiff2119_std))

In [ ]:
r2119_class_IR05 = RemapRange([[-1000000000, -10,"NODATA"],[-10,-5,-6],[-5,-2.5,-5],[-2.5,-2,-4],[-2,-1.5,-3],[-1.5,-1,-2],[-1,-0.5,-1],[-0.5,0.5,"NODATA"],[0.5,1,1],[1,1.5,2],[1.5,2,3],[2,2.5,4],[2.5,5,5],[5,10,6],[10, 1000000000,"NODATA"]])

r2119_class_IR04 = RemapRange([[-1000000000, -10,"NODATA"],[-10,-8,-13],[-8,-6,-12],[-6,-5,-11],[-5,-4,-10],[-4,-3.6,-9],[-3.6,-3.2,-8],
                               [-3.2,-2.8,-7],[-2.8,-2.4,-6],[-2.4,-2.0,-5],[-2.0,-1.6,-4],[-1.6,-1.2,-3],[-1.2,-0.8,-2],[-0.8,-0.4,-1],[-0.4,0.4,"NODATA"],
                               [0.4,0.8,1],[0.8,1.2,2],[1.2,1.6,3],[1.6,2.0,4],[2.0,2.4,5],[2.4,2.8,6],[2.8,3.2,7],[3.2,3.6,8],
                               [3.6,4.0,9],[4.0,5,10],[5,6,11],[6,8,12],[8,10,13],[10, 1000000000,"NODATA"]])

r2119_class_IR03 = RemapRange([[-1000000000, -10,"NODATA"],[-10,-7.5,-9],[-7.5,-5,-8],[-5,-2.5,-7],[-2.5,-1.5,-6],[-1.5,-1.25,-5],[-1.25,-1,-4],[-1,-0.75,-3],[-0.75,-0.5,-2],[-0.5,-0.25,-1],[-0.25,0.25,"NODATA"],
                               [0.25,0.5,1],[0.5,0.75,2],[0.75,1,3],[1,1.25,4],[1.25,1.5,5],[1.5,2.5,6],[2.5,5,7],[5,7.5,8],[7.5,10,9],[10, 1000000000,"NODATA"]])
                               
                               
r2119_class_IR02 = RemapRange([[-1000000000, -10,"NODATA"],[-10,-8,-13],[-8,-6,-12],[-6,-4,-11],[-4,-2,-10],[-2,-1.8,-9],[-1.8,-1.6,-8],[-1.6,-1.4,-7],[-1.4,-1.2,-6],[-1.2,-1,-5],[-1,-0.8,-4],[-0.8,-0.6,-3],[-0.6,-0.4,-2],[-0.4,-0.2,-1],[-0.2,0.2,"NODATA"],
                               [0.2,0.4,1],[0.4,0.6,2],[0.6,0.8,3],[0.8,1,4],[1,1.2,5],[1.2,1.4,6],[1.4,1.6,7],[1.6,1.8,8],[1.8,2,9],[2,4,10],[4,6,11],[6,8,12],[8,10,13],[10, 1000000000,"NODATA"]])
                               
r2119_class_IR01 = RemapRange([[-1000000000, -10,"NODATA"],[-10,-8,-14],[-8,-6,-13],[-6,-4,-12],[-4,-3.5,-11],[-3.5,-3,-10],[-3,-2.5,-9],[-2.5,-2,-8],[-2,-1.5,-7],[-1.5,-1,-6],[-1,-0.5,-5],[-0.5,-0.4,-4],[-0.4,-0.3,-3],[-0.3,-0.2,-2],[-0.2,-0.1,-1],[-0.1,0.1,"NODATA"],
                               [0.1,0.2,1],[0.2,0.3,2],[0.3,0.4,3],[0.4,0.5,4],[0.5,1,5],[1,1.5,6],[1.5,2,7],[2,2.5,8],[2.5,3,9],[3,3.5,10],[3.5,4,11],[4,6,12],[6,8,13],[8,10,14],[10, 1000000000,"NODATA"]])


In [ ]:
r2119_classes = [r2119_class_IR01, r2119_class_IR02, r2119_class_IR03, r2119_class_IR04, r2119_class_IR05]

In [ ]:
i = 0
for cl in r2119_classes:
    rc_rast2119 = Reclassify(r"{}\{}".format(folder_of_rast, mask_diff_2119), "Value", cl)
    rc_rast2119.save(r"{}\rc_rast2119_mapIR{}.tif".format(folder_of_rast, i))
    i += 1